In [ ]:
# Celda 1 — Cargar p5.js
from IPython.display import HTML
HTML("""
<script src="https://cdnjs.cloudflare.com/ajax/libs/p5.js/1.9.0/p5.min.js"></script>
<p style="color:#4caf50;font-weight:bold;">✅ p5.js cargado</p>
""")

In [2]:
# Celda 2 — SIMULACIÓN 1: Segunda Ley de Newton F = ma  (p. 123)
# calcForce() → updateAccel() → updateVelo() → moveObject()
from IPython.display import HTML

HTML("""
<div style="font-family:'Segoe UI',sans-serif; background:#0d1117; padding:18px;
            border-radius:12px; display:inline-block; color:#e0e0e0;">

  <!-- Título igual al screenshot -->
  <div style="background:#1c2333; padding:10px 16px; border-radius:8px;
              font-size:15px; font-weight:bold; margin-bottom:12px; color:#c9d1d9;">
    Página 123: Segunda Ley de Newton (F = ma)
  </div>

  <!-- Controles -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:10px; align-items:flex-end;">
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Fuerza (F)</div>
      <input type="range" id="f1_fuerza" min="0.05" max="1.0" step="0.05" value="0.15"
        style="width:110px;"
        oninput="document.getElementById('f1_fVal').innerText=parseFloat(this.value).toFixed(2)">
      <span id="f1_fVal" style="font-size:11px;margin-left:4px;">0.15</span>
    </div>
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Masa (m)</div>
      <input type="range" id="f1_masa" min="0.5" max="5" step="0.5" value="2"
        style="width:110px;"
        oninput="document.getElementById('f1_mVal').innerText=parseFloat(this.value).toFixed(1)">
      <span id="f1_mVal" style="font-size:11px;margin-left:4px;">2.0</span>
    </div>
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Drag (k)</div>
      <input type="range" id="f1_drag" min="0" max="0.08" step="0.005" value="0.01"
        style="width:110px;"
        oninput="document.getElementById('f1_kVal').innerText=parseFloat(this.value).toFixed(3)">
      <span id="f1_kVal" style="font-size:11px;margin-left:4px;">0.010</span>
    </div>
    <button onclick="f1Reset()"
      style="background:#e94560;color:#fff;border:none;padding:6px 14px;
             border-radius:7px;cursor:pointer;font-size:12px;font-weight:bold;">🔄</button>
  </div>

  <div id="f1-canvas"></div>

  <!-- Panel de datos — igual al screenshot -->
  <div id="f1-info"
    style="background:#161b22;border-radius:6px;padding:6px 12px;
           font-family:monospace;font-size:12px;color:#8b949e;margin-top:6px;">
  </div>
</div>

<script>
// ── Vector2D ────────────────────────────────────────────────
function Vec2(x,y){this.x=x;this.y=y;}
Vec2.prototype.add       = function(v){return new Vec2(this.x+v.x,this.y+v.y);};
Vec2.prototype.addScaled = function(v,s){return new Vec2(this.x+v.x*s,this.y+v.y*s);};
Vec2.prototype.multiply  = function(s){return new Vec2(this.x*s,this.y*s);};
Vec2.prototype.length    = function(){return Math.sqrt(this.x*this.x+this.y*this.y);};

// ── Forces (libro p.116-117) ────────────────────────────────
var Forces1 = {};
Forces1.constantGravity = function(m,g){ return new Vec2(0, m*g); };
Forces1.linearDrag      = function(k,v){ return v.multiply(-k); };
Forces1.add             = function(arr){
  var r = new Vec2(0,0);
  for(var i=0;i<arr.length;i++) r=r.add(arr[i]);
  return r;
};

var f1Sketch = function(p){
  var W=560, H=200, R=22;
  var pos, velo, acc, force, fGrav, fDrag;
  var trail=[], paused=false;

  function cfg(){
    return {
      F: parseFloat(document.getElementById('f1_fuerza').value),
      m: parseFloat(document.getElementById('f1_masa').value),
      k: parseFloat(document.getElementById('f1_drag').value)
    };
  }

  function reset(){
    pos   = new Vec2(60, 80);
    velo  = new Vec2(0, 0);
    acc   = new Vec2(0, 0);
    force = new Vec2(0, 0);
    fGrav = new Vec2(0, 0);
    fDrag = new Vec2(0, 0);
    trail = [];
  }
  window.f1Reset = reset;

  // ── Ciclo del libro (p.115/123) ─────────────────────────
  function calcForce(c){
    fGrav = Forces1.constantGravity(c.m, c.F);   // F_g = m*g (usamos F como g)
    fDrag = Forces1.linearDrag(c.k, velo);        // F_d = -k*v
    force = Forces1.add([fGrav, fDrag]);           // resultante
  }
  function updateAccel(c){ acc  = force.multiply(1/c.m); }   // a = F/m
  function updateVelo(){    velo = velo.addScaled(acc, 1); }  // v += a*dt
  function moveObject(){    pos  = pos.addScaled(velo, 1); }  // pos += v*dt

  function handleBounds(){
    if(pos.y > H-R){ pos.y=H-R; velo.y*=-0.7; }
    if(pos.y < R)  { pos.y=R;   velo.y*=-0.7; }
    if(pos.x > W-R){ pos.x=W-R; velo.x*=-0.85; }
    if(pos.x < R)  { pos.x=R;   velo.x*=-0.85; }
  }

  // Dibujar flecha
  function arrow(px,py,vx,vy,col,sc){
    sc=sc||1;
    var ex=px+vx*sc, ey=py+vy*sc;
    var len=Math.sqrt(vx*vx+vy*vy)*sc;
    if(len<2) return;
    p.stroke(col); p.strokeWeight(2.2);
    p.line(px,py,ex,ey);
    var ang=Math.atan2(ey-py,ex-px), hs=8;
    p.fill(col); p.noStroke();
    p.push(); p.translate(ex,ey); p.rotate(ang);
    p.triangle(0,0,-hs,-hs*0.4,-hs,hs*0.4);
    p.pop();
  }

  p.setup=function(){
    p.createCanvas(W,H).parent('f1-canvas');
    p.frameRate(60); reset();
  };

  p.draw=function(){
    p.background(13,17,23);
    // cuadrícula
    p.stroke(255,255,255,12); p.strokeWeight(1);
    for(var gx=0;gx<W;gx+=40) p.line(gx,0,gx,H);
    for(var gy=0;gy<H;gy+=40) p.line(0,gy,W,gy);
    // suelo
    p.stroke(233,69,96,140); p.strokeWeight(2);
    p.line(0,H-R,W,H-R);

    var c=cfg();
    calcForce(c); updateAccel(c); updateVelo(); moveObject(); handleBounds();
    trail.push({x:pos.x,y:pos.y});
    if(trail.length>100) trail.shift();

    // trayectoria
    for(var i=1;i<trail.length;i++){
      var a=p.map(i,0,trail.length,0,180);
      p.stroke(100,160,255,a); p.strokeWeight(1.5);
      p.line(trail[i-1].x,trail[i-1].y,trail[i].x,trail[i].y);
    }

    // vectores
    arrow(pos.x,    pos.y, 0,       fGrav.y, p.color(255,100,100), 60);
    arrow(pos.x,    pos.y, fDrag.x, fDrag.y, p.color(78,205,196),  60);
    arrow(pos.x+22, pos.y, force.x, force.y, p.color(255,230,109), 60);
    arrow(pos.x-22, pos.y, velo.x,  velo.y,  p.color(162,155,254), 18);

    // partícula
    p.noStroke(); p.fill(0,0,0,60);
    p.ellipse(pos.x+3,pos.y+3,R*2,R*2);
    p.stroke(180,210,255,160); p.strokeWeight(2);
    p.fill(100,160,255);
    p.ellipse(pos.x,pos.y,R*2,R*2);
    p.noStroke(); p.fill(255,255,255,100);
    p.ellipse(pos.x-R*0.3,pos.y-R*0.35,R*0.55,R*0.38);

    // info panel — igual al screenshot
    document.getElementById('f1-info').innerHTML =
      'Fuerza: ' + fGrav.y.toFixed(3) +
      ' | Masa: ' + c.m.toFixed(1) +
      ' | Accel: ' + acc.y.toFixed(3) +
      ' | Velocidad: (' + velo.x.toFixed(2) + ', ' + velo.y.toFixed(2) + ')';
  };
};
new p5(f1Sketch);
</script>
""")

In [3]:
# Celda 3 — SIMULACIÓN 2: Conservación del Momento — Colisión (p. 128)
# p = m*v  →  m1*v1 + m2*v2 = constante
from IPython.display import HTML

HTML("""
<div style="font-family:'Segoe UI',sans-serif; background:#0d1117; padding:18px;
            border-radius:12px; display:inline-block; color:#e0e0e0;">

  <!-- Título igual al screenshot -->
  <div style="background:#1c2333; padding:10px 16px; border-radius:8px;
              font-size:15px; font-weight:bold; margin-bottom:12px; color:#c9d1d9;">
    Página 128: Conservación del Momento (Impacto)
  </div>

  <!-- Controles -->
  <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:10px; align-items:flex-end;">
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Masa roja (m1)</div>
      <input type="range" id="p2_m1" min="0.5" max="5" step="0.5" value="2"
        style="width:100px;"
        oninput="document.getElementById('p2_m1Val').innerText=parseFloat(this.value).toFixed(1)">
      <span id="p2_m1Val" style="font-size:11px;margin-left:4px;">2.0</span>
    </div>
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Masa azul (m2)</div>
      <input type="range" id="p2_m2" min="0.5" max="5" step="0.5" value="1.5"
        style="width:100px;"
        oninput="document.getElementById('p2_m2Val').innerText=parseFloat(this.value).toFixed(1)">
      <span id="p2_m2Val" style="font-size:11px;margin-left:4px;">1.5</span>
    </div>
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Vel. roja (v1)</div>
      <input type="range" id="p2_v1" min="1" max="8" step="0.5" value="4"
        style="width:100px;"
        oninput="document.getElementById('p2_v1Val').innerText=parseFloat(this.value).toFixed(1)">
      <span id="p2_v1Val" style="font-size:11px;margin-left:4px;">4.0</span>
    </div>
    <div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:3px;">Elasticidad (e)</div>
      <input type="range" id="p2_e" min="0" max="1" step="0.05" value="1"
        style="width:100px;"
        oninput="document.getElementById('p2_eVal').innerText=parseFloat(this.value).toFixed(2)">
      <span id="p2_eVal" style="font-size:11px;margin-left:4px;">1.00</span>
    </div>
    <button onclick="p2Reset()"
      style="background:#e94560;color:#fff;border:none;padding:6px 14px;
             border-radius:7px;cursor:pointer;font-size:12px;font-weight:bold;">🔄</button>
  </div>

  <div id="p2-canvas"></div>

  <!-- Info panel -->
  <div id="p2-info"
    style="background:#161b22;border-radius:6px;padding:6px 12px;
           font-family:monospace;font-size:12px;color:#8b949e;margin-top:6px;
           line-height:1.7;">
  </div>

  <!-- Instrucción igual al screenshot -->
  <div style="margin-top:8px;font-size:12px;color:#555;">
    Presione <b style="color:#e94560;">CUALQUIER TECLA</b> para resetear todo
  </div>
</div>

<script>
var p2Sketch = function(p){
  var W=560, H=160;
  var b1, b2;          // las dos partículas
  var collided=false;
  var momentoAntes, momentoDespues;

  function cfg(){
    return {
      m1: parseFloat(document.getElementById('p2_m1').value),
      m2: parseFloat(document.getElementById('p2_m2').value),
      v1: parseFloat(document.getElementById('p2_v1').value),
      e:  parseFloat(document.getElementById('p2_e').value)
    };
  }

  function reset(){
    var c = cfg();
    var r1 = 14 + c.m1*4;   // radio proporcional a la masa
    var r2 = 14 + c.m2*4;
    b1 = { x:80,     y:H/2, vx:c.v1, vy:0, m:c.m1, r:r1, trail:[] };
    b2 = { x:W-100,  y:H/2, vx:0,    vy:0, m:c.m2, r:r2, trail:[] };
    collided=false;
    momentoAntes = b1.m*b1.vx + b2.m*b2.vx;
    momentoDespues = momentoAntes;
  }
  window.p2Reset = reset;

  // ── Colisión elástica/inelástica (Conservación del Momento) ──
  // Fórmulas del libro: m1*v1 + m2*v2 = m1*v1' + m2*v2'
  function checkCollision(){
    var dx = b2.x - b1.x;
    var dist = Math.abs(dx);
    if(dist < b1.r + b2.r && !collided){
      collided = true;
      var c  = cfg();
      var e  = c.e;         // coeficiente de restitución (1=elástico, 0=perfectamente inelástico)
      var m1 = b1.m, m2 = b2.m;
      var u1 = b1.vx, u2 = b2.vx;
      // Fórmulas de colisión con coef. de restitución:
      // v1' = ((m1-e*m2)*u1 + (1+e)*m2*u2) / (m1+m2)
      // v2' = ((m2-e*m1)*u2 + (1+e)*m1*u1) / (m1+m2)
      b1.vx = ((m1-e*m2)*u1 + (1+e)*m2*u2) / (m1+m2);
      b2.vx = ((m2-e*m1)*u2 + (1+e)*m1*u1) / (m1+m2);
      // Separar partículas para evitar overlap
      var overlap = (b1.r+b2.r - dist)/2;
      b1.x -= overlap; b2.x += overlap;
      momentoDespues = m1*b1.vx + m2*b2.vx;
    }
    // Rebote en paredes
    if(b1.x-b1.r < 0)  { b1.x=b1.r;   b1.vx=Math.abs(b1.vx); }
    if(b2.x+b2.r > W)  { b2.x=W-b2.r; b2.vx=-Math.abs(b2.vx); }
    if(b1.x+b1.r > b2.x-b2.r && collided){
      // segunda colisión: reiniciar flag
      collided=false;
    }
  }

  // Dibujar rastro
  function drawTrail(b, col){
    for(var i=1;i<b.trail.length;i++){
      var a=p.map(i,0,b.trail.length,0,160);
      p.stroke(p.red(col),p.green(col),p.blue(col),a);
      p.strokeWeight(p.map(i,0,b.trail.length,0.5,2.5));
      p.line(b.trail[i-1].x,H/2,b.trail[i].x,H/2);
    }
  }

  p.setup=function(){
    p.createCanvas(W,H).parent('p2-canvas');
    p.frameRate(60); reset();
  };

  p.keyPressed=function(){ reset(); };

  p.draw=function(){
    p.background(13,17,23);
    // cuadrícula
    p.stroke(255,255,255,10); p.strokeWeight(1);
    for(var gx=0;gx<W;gx+=40) p.line(gx,0,gx,H);
    // línea central
    p.stroke(255,255,255,20); p.strokeWeight(1);
    p.line(0,H/2,W,H/2);

    // mover
    b1.x += b1.vx; b2.x += b2.vx;
    checkCollision();

    // trails
    b1.trail.push({x:b1.x}); if(b1.trail.length>60) b1.trail.shift();
    b2.trail.push({x:b2.x}); if(b2.trail.length>60) b2.trail.shift();
    drawTrail(b1, p.color(255,100,100));
    drawTrail(b2, p.color(100,160,255));

    // ── partícula roja (b1) ──
    p.noStroke(); p.fill(0,0,0,50);
    p.ellipse(b1.x+3,H/2+3,b1.r*2,b1.r*2);
    // gradiente simulado
    p.fill(220,60,80); p.stroke(255,120,120); p.strokeWeight(2);
    p.ellipse(b1.x,H/2,b1.r*2,b1.r*2);
    p.noStroke(); p.fill(255,200,200,100);
    p.ellipse(b1.x-b1.r*0.3,H/2-b1.r*0.35,b1.r*0.55,b1.r*0.38);
    // etiqueta masa
    p.fill(255); p.noStroke(); p.textSize(10); p.textAlign(p.CENTER);
    p.text('m='+b1.m.toFixed(1), b1.x, H/2+b1.r+13);

    // ── partícula azul (b2) ──
    p.noStroke(); p.fill(0,0,0,50);
    p.ellipse(b2.x+3,H/2+3,b2.r*2,b2.r*2);
    p.fill(60,120,220); p.stroke(120,170,255); p.strokeWeight(2);
    p.ellipse(b2.x,H/2,b2.r*2,b2.r*2);
    p.noStroke(); p.fill(180,210,255,100);
    p.ellipse(b2.x-b2.r*0.3,H/2-b2.r*0.35,b2.r*0.55,b2.r*0.38);
    p.fill(255); p.noStroke(); p.textSize(10); p.textAlign(p.CENTER);
    p.text('m='+b2.m.toFixed(1), b2.x, H/2+b2.r+13);

    // flechas de velocidad
    if(Math.abs(b1.vx)>0.1){
      p.stroke(255,100,100); p.strokeWeight(2); p.fill(255,100,100);
      p.line(b1.x,H/2-b1.r-10, b1.x+b1.vx*8, H/2-b1.r-10);
      var d1=b1.vx>0?1:-1;
      p.triangle(b1.x+b1.vx*8,H/2-b1.r-10,
                 b1.x+b1.vx*8-d1*8,H/2-b1.r-14,
                 b1.x+b1.vx*8-d1*8,H/2-b1.r-6);
    }
    if(Math.abs(b2.vx)>0.1){
      p.stroke(100,160,255); p.strokeWeight(2); p.fill(100,160,255);
      p.line(b2.x,H/2-b2.r-10, b2.x+b2.vx*8, H/2-b2.r-10);
      var d2=b2.vx>0?1:-1;
      p.triangle(b2.x+b2.vx*8,H/2-b2.r-10,
                 b2.x+b2.vx*8-d2*8,H/2-b2.r-14,
                 b2.x+b2.vx*8-d2*8,H/2-b2.r-6);
    }

    // ── info panel ──
    var pTot = b1.m*b1.vx + b2.m*b2.vx;
    document.getElementById('p2-info').innerHTML =
      '<span style="color:#ff6b6b;">● Roja</span>:  ' +
      'v = ' + b1.vx.toFixed(2) + '  p = ' + (b1.m*b1.vx).toFixed(2) +
      ' &nbsp;&nbsp; ' +
      '<span style="color:#6eb4ff;">● Azul</span>:  ' +
      'v = ' + b2.vx.toFixed(2) + '  p = ' + (b2.m*b2.vx).toFixed(2) +
      '<br>' +
      'Momento total antes: <b style="color:#ffe66d;">' + momentoAntes.toFixed(3) + '</b>' +
      ' &nbsp;|&nbsp; después: <b style="color:#ffe66d;">' + momentoDespues.toFixed(3) + '</b>' +
      ' &nbsp;|&nbsp; actual: <b style="color:#4ecdc4;">' + pTot.toFixed(3) + '</b>' +
      (collided ? '' : '  ✅ conservado');
  };
};
new p5(p2Sketch);
</script>
""")